# Case Ambev — Long Neck NENO
**Papel:** Especialista de Planejamento → responder ao VP de Vendas e VP de Logística

**Perguntas do case:**
1. Devemos seguir com os incentivos comerciais?
2. Qual o plano de produção e transferência?
3. Quanto vai custar?
4. Quais os riscos?

---
**Premissas:**
- AQ541 (Aquiraz/CE): cap. **12.600 hl/semana**
- NS541 (Pernambuco/PE): cap. **27.000 hl/semana**
- Cabotagem SP→NE: lead time **25 dias**
- Rodoviário: lead time **6 dias**, custo **+60%**, risco avaria **+5%**
- DOI mínimo NENO: **12 dias**
- BIAS histórico das GEOs: **+9%** (superestimam demanda)

In [11]:
import pandas as pd
import numpy as np

pd.set_option('display.float_format', '{:,.1f}'.format)
pd.set_option('display.max_columns', 20)
pd.set_option('display.width', 130)

semanas = ['W0_02fev', 'W1_09fev', 'W2_16fev', 'W3_23fev']
DOI_MIN = 12
CAP_AQ  = 12600
CAP_NS  = 27000

## 1. PCP — Produção Sugerida e Folgas Disponíveis

In [12]:
# AQ541 — Aquiraz/CE
pcp_aq = pd.DataFrame({
    'sku':      ['MALZBIER BRAHMA', 'PATAGONIA AMBER LAGER', 'COLORADO LAGER'],
    'W0_02fev': [0,     12240, 0],
    'W1_09fev': [9000,  1800,  0],
    'W2_16fev': [7560,  5040,  0],
    'W3_23fev': [0,     12600, 0],
})

# NS541 — Pernambuco/PE
pcp_ns = pd.DataFrame({
    'sku':      ['BRAHMA CHOPP ZERO', 'GOOSE ISLAND MIDWAY', 'MALZBIER BRAHMA',
                 'COLORADO LAGER', 'SKOL BEATS SENSES', 'BUDWEISER ZERO'],
    'W0_02fev': [0,    5400,  16200, 5400, 0,    0    ],
    'W1_09fev': [0,    14400, 0,     0,    0,    5400 ],
    'W2_16fev': [0,    0,     12960, 10800,3240, 0    ],
    'W3_23fev': [3600, 12600, 0,     0,    0,    10800],
})

# Folgas por semana
folgas = pd.DataFrame({'semana': semanas})
folgas['total_aq']    = [pcp_aq[s].sum()  for s in semanas]
folgas['folga_aq']    = CAP_AQ - folgas['total_aq']
folgas['total_ns']    = [pcp_ns[s].sum()  for s in semanas]
folgas['folga_ns']    = CAP_NS - folgas['total_ns']
folgas['folga_total'] = folgas['folga_aq'] + folgas['folga_ns']

print('=== AQ541 — Programação PCP ===')
display(pcp_aq)
print()
print('=== NS541 — Programação PCP ===')
display(pcp_ns)
print()
print('=== FOLGAS DISPONÍVEIS POR SEMANA (hl) ===')
display(folgas)
print(f"\n📌 Folga total: {folgas['folga_total'].sum():,.0f} hl")
print(f"   W0: {folgas.loc[0,'folga_total']:,.0f} hl (só AQ)")
print(f"   W1: {folgas.loc[1,'folga_total']:,.0f} hl (1.800 AQ + 7.200 NS) ← maior janela")
print(f"   W2 e W3: zero folga — linhas 100% ocupadas")

=== AQ541 — Programação PCP ===


,sku,W0_02fev,W1_09fev,W2_16fev,W3_23fev
0,MALZBIER BRAHMA,0,9000,7560,0
1,PATAGONIA AMBER LAGER,12240,1800,5040,12600
2,COLORADO LAGER,0,0,0,0



=== NS541 — Programação PCP ===


,sku,W0_02fev,W1_09fev,W2_16fev,W3_23fev
0,BRAHMA CHOPP ZERO,0,0,0,3600
1,GOOSE ISLAND MIDWAY,5400,14400,0,12600
2,MALZBIER BRAHMA,16200,0,12960,0
3,COLORADO LAGER,5400,0,10800,0
4,SKOL BEATS SENSES,0,0,3240,0
5,BUDWEISER ZERO,0,5400,0,10800



=== FOLGAS DISPONÍVEIS POR SEMANA (hl) ===


,semana,total_aq,folga_aq,total_ns,folga_ns,folga_total
0,W0_02fev,12240,360,27000,0,360
1,W1_09fev,10800,1800,19800,7200,9000
2,W2_16fev,12600,0,27000,0,0
3,W3_23fev,12600,0,27000,0,0



📌 Folga total: 9,360 hl
   W0: 360 hl (só AQ)
   W1: 9,000 hl (1.800 AQ + 7.200 NS) ← maior janela
   W2 e W3: zero folga — linhas 100% ocupadas


## 2. Demanda Malzbier — Nova Demanda (+30%)

In [13]:
malzbier = pd.DataFrame({
    'geo':      ['Mapapi', 'NE Norte', 'NE Sul', 'NO Araguaia', 'NO Centro'],
    'ei_w0':    [1985.6,   302.0,      4383.0,   0.0,           964.8     ],
    'W0_02fev': [6074.8,   1971.8,     3230.3,   78.9,          2227.5    ],
    'W1_09fev': [6286.4,   2265.8,     3517.6,   92.9,          2463.1    ],
    'W2_16fev': [4258.0,   1707.9,     2589.1,   62.9,          1789.6    ],
    'W3_23fev': [5204.6,   1844.0,     2857.3,   65.9,          2025.9    ],
})
malzbier['total_mes'] = malzbier[semanas].sum(axis=1)

dem_semana = [malzbier[s].sum() for s in semanas]
ei_total   = malzbier['ei_w0'].sum()
dem_total  = malzbier['total_mes'].sum()

display(malzbier)
print(f"\nDemanda total (nova, +30%):          {dem_total:>10,.1f} HL")
print(f"Demanda ajustada pelo BIAS (-9%):    {dem_total/1.09:>10,.1f} HL")
print(f"Estoque inicial total:               {ei_total:>10,.1f} HL")
print(f"\n🚨 NE Norte: EI = 302 HL — menos de 1 dia de estoque. Risco crítico de ruptura na W0!")

,geo,ei_w0,W0_02fev,W1_09fev,W2_16fev,W3_23fev,total_mes
0,Mapapi,"1,985.6","6,074.8","6,286.4","4,258.0","5,204.6","21,823.8"
1,NE Norte,302.0,"1,971.8","2,265.8","1,707.9","1,844.0","7,789.5"
2,NE Sul,"4,383.0","3,230.3","3,517.6","2,589.1","2,857.3","12,194.3"
3,NO Araguaia,0.0,78.9,92.9,62.9,65.9,300.6
4,NO Centro,964.8,"2,227.5","2,463.1","1,789.6","2,025.9","8,506.1"



Demanda total (nova, +30%):            50,614.3 HL
Demanda ajustada pelo BIAS (-9%):      46,435.1 HL
Estoque inicial total:                  7,635.4 HL

🚨 NE Norte: EI = 302 HL — menos de 1 dia de estoque. Risco crítico de ruptura na W0!


## 3. Balanço Semana a Semana — Com Folgas — Malzbier

> **DOI prospectivo** = EF / dem_semana_seguinte × 6  
> **Demanda coberta com BIAS −9%** (GEOs sistematicamente superestimam em +9%).  
> Folgas de produção já incluídas na produção total.  
> Balanço mostra a situação real **antes** de qualquer transferência externa.


In [18]:
semanas   = ['W0_02fev', 'W1_09fev', 'W2_16fev', 'W3_23fev']
DOI_MIN   = 12
AVARIA    = 0.95
BIAS      = 1.09

pcp_malz  = [
    pcp_aq.loc[pcp_aq['sku']=='MALZBIER BRAHMA', s].values[0] +
    pcp_ns.loc[pcp_ns['sku']=='MALZBIER BRAHMA', s].values[0]
    for s in semanas
]
folga_sem = folgas['folga_total'].tolist()

# Demanda total bruta por semana (soma dos GEOs)
dem_bruta  = [malzbier[s].sum() for s in semanas]
# Demanda ajustada pelo BIAS de +9%
dem_semana = [d / BIAS for d in dem_bruta]

print('─── COM FOLGAS — demanda com BIAS −9% ───')
print('Folga embutida na produção (já descontada do que precisa vir de externo)\n')

estoque = ei_total
rows = []
for i, s in enumerate(semanas):
    prod     = pcp_malz[i] + folga_sem[i]
    dem      = dem_semana[i]
    dem_prox = dem_semana[i+1] if i < 3 else dem_semana[i]
    ef       = estoque + prod - dem
    doi      = ef / dem_prox * 6
    rows.append({
        'semana':        s,
        'EI (hl)':       round(estoque, 0),
        'prod + folga':  round(prod, 0),
        'dem c/BIAS':    round(dem, 0),
        'EF (hl)':       round(ef, 0),
        'DOI (dias)':    round(doi, 1),
        'status':        '✅' if doi >= DOI_MIN else '🚨'
    })
    estoque = ef

display(pd.DataFrame(rows))

abaixo = [(r['semana'][:2], r['DOI (dias)']) for r in rows if r['DOI (dias)'] < DOI_MIN]
print()
print('📌 CONCLUSÃO:')
print(f'   Mesmo com 100% das folgas ({sum(folga_sem):,.0f} hl), DOI < 12d em: '
      + '  '.join([f"{s} ({d:.1f}d)" for s,d in abaixo]))
print('   Transferência externa obrigatória — rodo W0-W2, cabotagem W3 se precisar.')


─── COM FOLGAS — demanda com BIAS −9% ───
Folga embutida na produção (já descontada do que precisa vir de externo)



,semana,EI (hl),prod + folga,dem c/BIAS,EF (hl),DOI (dias),status
0,W0_02fev,"7,635.0",16560,"12,462.0","11,734.0",5.2,🚨
1,W1_09fev,"11,734.0",18000,"13,418.0","16,315.0",10.3,🚨
2,W2_16fev,"16,315.0",20520,"9,548.0","27,287.0",14.9,✅
3,W3_23fev,"27,287.0",0,"11,007.0","16,280.0",8.9,🚨



📌 CONCLUSÃO:
   Mesmo com 100% das folgas (9,360 hl), DOI < 12d em: W0 (5.2d)  W1 (10.3d)  W3 (8.9d)
   Transferência externa obrigatória — rodo W0-W2, cabotagem W3 se precisar.


## 4. Volume de Transferência Externo — Todas as LN

> **Modais disponíveis (data-base: 02/02):**
> - **W0 (02-08/02):** RODO — lead 6d, embarca hoje, chega dia 8 (ainda em W0, antes de W1 dia 9) ✅
> - **W1 (09-15/02):** RODO — lead 6d
> - **W2 (16-22/02):** RODO — lead 6d
> - **W3 (23-28/02):** CABOTAGEM — lead 25d, embarca hoje dia 2, chega dia 27 (dentro de W3) ✅ — mais barato que rodo
>
> **Custo:** rodo = cabo × 1,6 | avaria rodo = 5% (embarque = chegada / 0,95)
> **BIAS:** −9% em Malzbier e Goose. Colorado já com DOI ≥ 12d — sem transferência.
> **Folgas:** já embutidas no EF — não há volume extra a descontar na transferência.
> **NO Araguaia:** excluído do split (retirada em UB541/MG).
> **Patagonia:** sem rota de transferência SE→NENO — apenas alerta de DOI.


In [17]:
XLS = 'Analise_LongNeck_WSNP - Sem repostas.xlsx'
df_nd = pd.read_excel(XLS, sheet_name='Cenário com Nova Demanda', header=None)
df_ct = pd.read_excel(XLS, sheet_name='Custos de transferência',  header=None)

# ── Mapa de colunas ──────────────────────────────────────────────────────────
_SC = {
    'W0': {'dem': 3,  'wsnp':  5, 'ei':  7, 'ef': 13},
    'W1': {'dem': 16, 'wsnp': 18,            'ef': 24},
    'W2': {'dem': 27, 'wsnp': 29,            'ef': 35},
    'W3': {'dem': 38, 'wsnp': 40,            'ef': 46},
}
_SEMS     = ['W0', 'W1', 'W2', 'W3']
_ALL_GEOS = ['Mapapi', 'NE Norte', 'NE Sul', 'NO Araguaia', 'NO Centro']
_CDR_GEO  = {'Mapapi': 'PB', 'NE Norte': 'PB', 'NE Sul': 'BA', 'NO Centro': 'PB'}

# Custos cabo (planilha) e rodo (cabo × 1,6)
_CABO = {
    ('COLORADO','BA'): df_ct.iloc[3,4], ('COLORADO','PB'): df_ct.iloc[4,4],
    ('GOOSE',   'BA'): df_ct.iloc[5,4], ('GOOSE',   'PB'): df_ct.iloc[7,4],
    ('MALZBIER','BA'): df_ct.iloc[6,4], ('MALZBIER','PB'): df_ct.iloc[8,4],
}

def modal_custo(sku, cdr, sem):
    """W3 → cabotagem (cabo); W0-W2 → rodo (cabo × 1,6)"""
    cabo = _CABO[(sku, cdr)]
    return (cabo, 'CABO') if sem == 'W3' else (cabo * 1.6, 'RODO')

def get_prop_ba(gs, sem):
    dem_ba = sum(df_nd.iloc[gs+j, _SC[sem]['dem']] for j,g in enumerate(_ALL_GEOS)
                 if g in _CDR_GEO and _CDR_GEO[g]=='BA')
    dem_pb = sum(df_nd.iloc[gs+j, _SC[sem]['dem']] for j,g in enumerate(_ALL_GEOS)
                 if g in _CDR_GEO and _CDR_GEO[g]=='PB')
    return dem_ba / (dem_ba + dem_pb)

# ── helper: calcula transferência de um SKU ───────────────────────────────────
def calc_transferencia(sku, gs, ti, use_bias, use_ef_excel=False):
    """
    use_bias     : aplica BIAS −9% na demanda (Malzbier, Goose)
    use_ef_excel : usa EF do Excel como baseline (Goose/Colorado/Patagonia)
                   False → calcula sequencialmente (Malzbier, que não tem transf. internas)
    """
    cumul_rodo = 0.
    t_ba_acc = t_pb_acc = c_ba_acc = c_pb_acc = 0.
    rows = []
    # estoque inicial (só usado se use_ef_excel=False)
    estoque = df_nd.iloc[ti, _SC['W0']['ei']]
    # produção e folga (só Malzbier usa folga explícita)
    if sku == 'MALZBIER':
        prod_list = [p + f for p,f in zip(pcp_malz, folga_sem)]
    else:
        prod_list = [0,0,0,0]  # produção já embutida no EF do Excel

    for i, sem in enumerate(_SEMS):
        dem_bruta = df_nd.iloc[ti, _SC[sem]['dem']]
        dem       = dem_bruta / BIAS if use_bias else dem_bruta
        dem_prox_bruta = df_nd.iloc[ti, _SC[_SEMS[i+1]]['dem']] if i < 3 else dem_bruta
        dem_prox  = dem_prox_bruta / BIAS if use_bias else dem_prox_bruta

        if use_ef_excel:
            ef_excel = df_nd.iloc[ti, _SC[sem]['ef']]
            # ajuste: EF do excel usa dem_bruta; com BIAS a dem é menor → EF maior
            ef_st = ef_excel - dem_bruta + dem + cumul_rodo
        else:
            ef_st = estoque + prod_list[i] - dem

        doi_st  = ef_st / dem_prox * 6
        ef_min  = DOI_MIN * dem_prox / 6
        transf  = max(0., ef_min - ef_st)
        prop_ba = get_prop_ba(gs, sem)
        custo_ba, modal = modal_custo(sku, 'BA', sem)
        custo_pb, _     = modal_custo(sku, 'PB', sem)
        t_ba_emb = transf * prop_ba       / AVARIA
        t_pb_emb = transf * (1 - prop_ba) / AVARIA
        t_ba_acc += t_ba_emb; t_pb_acc += t_pb_emb
        c_ba_acc += t_ba_emb * custo_ba;  c_pb_acc += t_pb_emb * custo_pb
        cumul_rodo += transf
        ef_final   = ef_st + transf
        doi_final  = ef_final / dem_prox * 6
        flag       = '✅' if doi_st >= DOI_MIN else '🚨'
        rows.append({'Semana': sem, 'Modal': modal,
                     'EF s/ext': round(ef_st,0), 'DOI s/ext': round(doi_st,1),
                     'flag': flag, 'Transf.chegada': round(transf,0),
                     'Emb.BA': round(t_ba_emb,0), 'Emb.PB': round(t_pb_emb,0),
                     'EF final': round(ef_final,0), 'DOI final': round(doi_final,1)})
        if not use_ef_excel:
            estoque = ef_final

    return rows, t_ba_acc, t_pb_acc, c_ba_acc, c_pb_acc


# ── MALZBIER ─────────────────────────────────────────────────────────────────
print('=' * 68)
print('MALZBIER — BIAS −9% | Folgas incluídas | W0-W2 rodo | W3 cabo')
print('=' * 68)
r_m, ba_m, pb_m, cba_m, cpb_m = calc_transferencia('MALZBIER', 20, 25, True, False)
display(pd.DataFrame(r_m).rename(columns={'flag':'⚑'}))
rodo_ba_malz = _CABO[('MALZBIER','BA')] * 1.6
rodo_pb_malz = _CABO[('MALZBIER','PB')] * 1.6
cabo_ba_malz = _CABO[('MALZBIER','BA')]
cabo_pb_malz = _CABO[('MALZBIER','PB')]
print(f'  Custo unitário → Rodo BA: R${rodo_ba_malz:.2f}/HL | Rodo PB: R${rodo_pb_malz:.2f}/HL'
      f' | Cabo BA: R${cabo_ba_malz:.2f}/HL | Cabo PB: R${cabo_pb_malz:.2f}/HL')
print(f'  CDR Camaçari (BA):    {ba_m:>7,.0f} HL  →  R${cba_m:>10,.0f}')
print(f'  CDR João Pessoa (PB): {pb_m:>7,.0f} HL  →  R${cpb_m:>10,.0f}')
print(f'  SUBTOTAL MALZBIER:    {ba_m+pb_m:>7,.0f} HL  →  R${cba_m+cpb_m:>10,.0f}')

# ── GOOSE ─────────────────────────────────────────────────────────────────────
print()
print('=' * 68)
print('GOOSE — BIAS −9% | EF baseline do Excel | W0-W2 rodo | W3 cabo')
print('=' * 68)
r_g, ba_g, pb_g, cba_g, cpb_g = calc_transferencia('GOOSE', 12, 17, True, True)
display(pd.DataFrame(r_g).rename(columns={'flag':'⚑'}))
rodo_ba_goose = _CABO[('GOOSE','BA')] * 1.6
rodo_pb_goose = _CABO[('GOOSE','PB')] * 1.6
cabo_ba_goose = _CABO[('GOOSE','BA')]
cabo_pb_goose = _CABO[('GOOSE','PB')]
print(f'  Custo unitário → Rodo BA: R${rodo_ba_goose:.2f}/HL | Rodo PB: R${rodo_pb_goose:.2f}/HL'
      f' | Cabo BA: R${cabo_ba_goose:.2f}/HL | Cabo PB: R${cabo_pb_goose:.2f}/HL')
print(f'  CDR Camaçari (BA):    {ba_g:>7,.0f} HL  →  R${cba_g:>10,.0f}')
print(f'  CDR João Pessoa (PB): {pb_g:>7,.0f} HL  →  R${cpb_g:>10,.0f}')
print(f'  SUBTOTAL GOOSE:       {ba_g+pb_g:>7,.0f} HL  →  R${cba_g+cpb_g:>10,.0f}')

# ── COLORADO ─────────────────────────────────────────────────────────────────
print()
print('=' * 68)
print('COLORADO — DOI ≥ 12d em todas as semanas (sem BIAS necessário)')
print('=' * 68)
rows_co = []
for i, sem in enumerate(_SEMS):
    ef  = df_nd.iloc[33, _SC[sem]['ef']]
    dp  = df_nd.iloc[33, _SC[_SEMS[i+1]]['dem']] if i < 3 else df_nd.iloc[33, _SC[sem]['dem']]
    doi = ef / dp * 6
    rows_co.append({'Semana': sem, 'EF': round(ef,0), 'DOI (d)': round(doi,1),
                    'Status': '✅ ok' if doi >= DOI_MIN else '🚨'})
display(pd.DataFrame(rows_co))
print('  ✅ Sem transferência necessária.')

# ── PATAGONIA ─────────────────────────────────────────────────────────────────
print()
print('=' * 68)
print('PATAGONIA — sem rota de transferência SE→NENO')
print('Produzida exclusivamente no NENO (AQ541) — apenas monitoramento de DOI')
print('=' * 68)
rows_pat = []
for i, sem in enumerate(_SEMS):
    ef  = df_nd.iloc[9, _SC[sem]['ef']]
    dp  = df_nd.iloc[9, _SC[_SEMS[i+1]]['dem']] if i < 3 else df_nd.iloc[9, _SC[sem]['dem']]
    doi = ef / dp * 6
    flag = '✅' if doi >= DOI_MIN else '⚠️  <12d — ação via malha interna'
    rows_pat.append({'Semana': sem, 'EF': round(ef,0), 'DOI (d)': round(doi,1), 'Status': flag})
display(pd.DataFrame(rows_pat))
print('  ⚠️  Não consta rota de custo SE→NENO para Patagonia.')


MALZBIER — BIAS −9% | Folgas incluídas | W0-W2 rodo | W3 cabo


,Semana,Modal,EF s/ext,DOI s/ext,⚑,Transf.chegada,Emb.BA,Emb.PB,EF final,DOI final
0,W0,RODO,"11,734.0",5.2,🚨,"15,103.0","3,803.0","12,095.0","26,836.0",12.0
1,W1,RODO,"31,418.0",19.7,✅,0.0,0.0,0.0,"31,418.0",19.7
2,W2,RODO,"42,390.0",23.1,✅,0.0,0.0,0.0,"42,390.0",23.1
3,W3,CABO,"31,383.0",17.1,✅,0.0,0.0,0.0,"31,383.0",17.1


  Custo unitário → Rodo BA: R$135.33/HL | Rodo PB: R$152.53/HL | Cabo BA: R$84.58/HL | Cabo PB: R$95.33/HL
  CDR Camaçari (BA):      3,803 HL  →  R$   514,618
  CDR João Pessoa (PB):  12,095 HL  →  R$ 1,844,742
  SUBTOTAL MALZBIER:     15,897 HL  →  R$ 2,359,360

GOOSE — BIAS −9% | EF baseline do Excel | W0-W2 rodo | W3 cabo


,Semana,Modal,EF s/ext,DOI s/ext,⚑,Transf.chegada,Emb.BA,Emb.PB,EF final,DOI final
0,W0,RODO,"21,435.0",9.1,🚨,"6,967.0","1,597.0","5,737.0","28,402.0",12.0
1,W1,RODO,"27,270.0",15.2,✅,0.0,0.0,0.0,"27,270.0",15.2
2,W2,RODO,"15,881.0",8.0,🚨,"8,063.0","1,751.0","6,736.0","23,944.0",12.0
3,W3,CABO,"38,709.0",19.4,✅,0.0,0.0,0.0,"38,709.0",19.4


  Custo unitário → Rodo BA: R$131.83/HL | Rodo PB: R$141.28/HL | Cabo BA: R$82.40/HL | Cabo PB: R$88.30/HL
  CDR Camaçari (BA):      3,348 HL  →  R$   441,368
  CDR João Pessoa (PB):  12,473 HL  →  R$ 1,762,115
  SUBTOTAL GOOSE:        15,821 HL  →  R$ 2,203,483

COLORADO — DOI ≥ 12d em todas as semanas (sem BIAS necessário)


,Semana,EF,DOI (d),Status
0,W0,"20,738.0",18.5,✅ ok
1,W1,"13,797.0",16.5,✅ ok
2,W2,"19,218.0",20.5,✅ ok
3,W3,"13,213.0",14.1,✅ ok


  ✅ Sem transferência necessária.

PATAGONIA — sem rota de transferência SE→NENO
Produzida exclusivamente no NENO (AQ541) — apenas monitoramento de DOI


,Semana,EF,DOI (d),Status
0,W0,"20,189.0",13.1,✅
1,W1,"12,226.0",10.8,⚠️ <12d — ação via malha interna
2,W2,"10,428.0",8.4,⚠️ <12d — ação via malha interna
3,W3,"14,483.0",11.7,⚠️ <12d — ação via malha interna


  ⚠️  Não consta rota de custo SE→NENO para Patagonia.


## 5. Custos e MACO — Produção Local vs Transferência


In [16]:
# Custos unitários
cabo_ba_malz = _CABO[('MALZBIER','BA')];  cabo_pb_malz = _CABO[('MALZBIER','PB')]
rodo_ba_malz = cabo_ba_malz * 1.6;        rodo_pb_malz = cabo_pb_malz * 1.6
cabo_ba_goose = _CABO[('GOOSE','BA')];    cabo_pb_goose = _CABO[('GOOSE','PB')]
rodo_ba_goose = cabo_ba_goose * 1.6;      rodo_pb_goose = cabo_pb_goose * 1.6

maco_malz  = 285.0   # R$/HL produção local
maco_goose = 350.0
maco_color = 300.0

print('=== MACO LÍQUIDO (MACO produção local − custo frete) ===')
print(f'  Malzbier  → local: R${maco_malz:.2f}/HL  | rodo BA: R${maco_malz-rodo_ba_malz:.2f}/HL'
      f'  | rodo PB: R${maco_malz-rodo_pb_malz:.2f}/HL'
      f'  | cabo BA: R${maco_malz-cabo_ba_malz:.2f}/HL'
      f'  | cabo PB: R${maco_malz-cabo_pb_malz:.2f}/HL')
print(f'  Goose     → local: R${maco_goose:.2f}/HL  | rodo BA: R${maco_goose-rodo_ba_goose:.2f}/HL'
      f'  | rodo PB: R${maco_goose-rodo_pb_goose:.2f}/HL'
      f'  | cabo BA: R${maco_goose-cabo_ba_goose:.2f}/HL'
      f'  | cabo PB: R${maco_goose-cabo_pb_goose:.2f}/HL')
print()

# Consolidado
tot_hl_malz  = ba_m + pb_m
tot_r_malz   = cba_m + cpb_m
tot_hl_goose = ba_g + pb_g
tot_r_goose  = cba_g + cpb_g
tot_hl = tot_hl_malz + tot_hl_goose
tot_r  = tot_r_malz  + tot_r_goose

print('=' * 68)
print('CONSOLIDADO FINAL — TRANSFERÊNCIA RODO SP → NENO')
print('Data-base 02/02 | W0-W2: rodo | BIAS −9% Malzbier e Goose')
print('=' * 68)
print(f'{"SKU":<12}  {"HL total":>9}  {"Custo total":>13}  Observação')
print('-' * 68)
print(f'{"MALZBIER":<12}  {tot_hl_malz:>9,.0f}  R${tot_r_malz:>11,.0f}  W0 rodo + W3 cabo')
print(f'{"GOOSE":<12}  {tot_hl_goose:>9,.0f}  R${tot_r_goose:>11,.0f}  W0 e W2 rodo + W3 cabo')
print(f'{"COLORADO":<12}  {"—":>9}  {"—":>13}  DOI ≥ 12d — sem transferência')
print(f'{"PATAGONIA":<12}  {"—":>9}  {"—":>13}  Sem rota SE→NENO')
print('-' * 68)
print(f'{"TOTAL":<12}  {tot_hl:>9,.0f}  R${tot_r:>11,.0f}')
print()
cba_total = cba_m + cba_g
cpb_total = cpb_m + cpb_g
ba_total  = ba_m  + ba_g
pb_total  = pb_m  + pb_g
print(f'  CDR Camaçari (BA):    {ba_total:>7,.0f} HL  →  R${cba_total:>10,.0f}')
print(f'  CDR João Pessoa (PB): {pb_total:>7,.0f} HL  →  R${cpb_total:>10,.0f}')
print()
print('   Rodo W0: embarcar HOJE para chegar até 08/02 (antes de W1).')
print('   Acionar cabotagem também para março — lead 25d inviabiliza reação tardia.')


=== MACO LÍQUIDO (MACO produção local − custo frete) ===
  Malzbier  → local: R$285.00/HL  | rodo BA: R$149.67/HL  | rodo PB: R$132.47/HL  | cabo BA: R$200.42/HL  | cabo PB: R$189.67/HL
  Goose     → local: R$350.00/HL  | rodo BA: R$218.17/HL  | rodo PB: R$208.72/HL  | cabo BA: R$267.60/HL  | cabo PB: R$261.70/HL

CONSOLIDADO FINAL — TRANSFERÊNCIA RODO SP → NENO
Data-base 02/02 | W0-W2: rodo | BIAS −9% Malzbier e Goose
SKU            HL total    Custo total  Observação
--------------------------------------------------------------------
MALZBIER         15,897  R$  2,359,360  W0 rodo + W3 cabo
GOOSE            15,821  R$  2,203,483  W0 e W2 rodo + W3 cabo
COLORADO              —              —  DOI ≥ 12d — sem transferência
PATAGONIA             —              —  Sem rota SE→NENO
--------------------------------------------------------------------
TOTAL            31,718  R$  4,562,843

  CDR Camaçari (BA):      7,151 HL  →  R$   955,986
  CDR João Pessoa (PB):  24,567 HL  →  R$ 3,606,

## 6. Resumo 

| Dimensão | Ponto crítico |
|---|---|
| **Malzbier** | DOI 5.2d em W0 — rodo obrigatório hoje |
| **Goose** | DOI 9.1d em W0 e 8.0d em W2 — rodo em W0 e W2 |
| **Colorado** | DOI ≥ 12d — nenhuma ação necessária |
| **Patagonia** | DOI <12d em W1-W3 — sem rota externa |
